In [ ]:
# pip install transformer_lens einops
from huggingface_hub import login
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
from transformer_lens import HookedTransformer
import torch
from einops import rearrange, repeat, einsum
import matplotlib.pyplot as plt
from math import sqrt

In [ ]:

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

model = HookedTransformer.from_pretrained("google/gemma-2-2b", device=device, center_unembed=False, cache_dir="local_files/gemma-2-2b")
model.eval()

In [ ]:
prompt = "The capital of France is"
prompt = "La capitale della Francia è"
prompt = "La capitale della Francia è Parigi, e la città è famosa per la sua Torre Eiffel, il Louvre e la sua ricca storia culturale."
prompt = "The capital of France is Paris, and the city is famous for its Eiffel Tower, the Louvre, and its rich cultural history."
#prompt = "Solomon Grundy born on a Monday, christened on Tuesday, married on Wednesday, took ill on Thursday, worse on Friday, died on Saturday, buried on Sunday. The end."
tokens = model.to_tokens(prompt)

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

qs, ks = {}, {}
for layer in range(model.cfg.n_layers):
    qs[layer] = cache[f"blocks.{layer}.attn.hook_q"][0]  # (seq, n_heads, d_head)
    ks[layer] = cache[f"blocks.{layer}.attn.hook_k"][0]  # (seq, n_kv_heads, d_head)

In [ ]:
fanos = []
var_mutuals = []
mutuals = []
for layer in qs.keys():
    q = qs[layer]
    k = ks[layer]
    n_q_per_kv = q.shape[1] // k.shape[1]
    k = k.repeat_interleave(n_q_per_kv, dim=1)  # (seq, n_heads, d_head)

    qk = torch.cat([q, k], dim=0)  # (2*seq, d_head)

    qk_mean = qk.mean(dim=0, keepdim=True)
    qk_centered = (qk - qk_mean)
    qk_cov = einsum(qk_centered, qk_centered, "t1 h d,t2 h d->h t1 t2") / qk_centered.shape[-1]
    S_11 = qk_cov[:, :q.shape[0], :q.shape[0]] 
    S_12 = qk_cov[:, :q.shape[0], q.shape[0]:]
    S_22 = qk_cov[:, q.shape[0]:, q.shape[0]:]

    S_11_inv = torch.linalg.inv(S_11 + 1e-6 * torch.eye(S_11.shape[-1], device=S_11.device))
    S_22_inv = torch.linalg.inv(S_22 + 1e-6 * torch.eye(S_22.shape[-1], device=S_22.device))
        

    
    v_matr = S_11_inv @ S_12 @ S_22_inv @ S_12.transpose(-2, -1)
    v_matr_values = torch.linalg.eigvalsh(v_matr)
    v_matr_values = torch.clamp(v_matr_values, min=0.0, max=1.0 - 1e-6)

    var_mutual = torch.sum(v_matr_values, dim=-1)

    mutual =-0.5 * torch.log(1.0 - v_matr_values).sum(dim=-1)
    fano = var_mutual / mutual
    fanos.append(fano.cpu())
    var_mutuals.append(var_mutual.cpu())
    mutuals.append(mutual.cpu())
fanos = torch.stack(fanos)
var_mutuals = torch.stack(var_mutuals)
mutuals = torch.stack(mutuals)

In [ ]:
plt.imshow(torch.exp(torch.log(var_mutuals) - torch.log(mutuals)), vmin=0)
plt.colorbar(label="log(Var[Mutual Info]) - log(Mutual Info)")
plt.title("Fano Factor of Mutual Information")
plt.xlabel("Head")
plt.ylabel("Layer")
plt.show()